# 2.5 記憶範圍怎麼影響預測？

# 第 2 章：讓模型看更長的上下文

前置：第1章的索引、loss、梯度。這章只增加固定上下文，沒有 attention。向量是可以學習的數字列表，不是先驗語意標籤。




In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：上下文像把幾張卡片一起攤開**

拼接順序保留位置；窗口固定長度。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/context.svg")))

**先想一想：**「紅色物體是圓」「藍色物體是方」，只看到「物體是」能答嗎？

如果答案取決於三格前的線索，只看最後一格的模型無法分辨兩個問題。先檢查資料可辨識性，再怪模型不會學。

**把直覺寫成數學：**若兩例的可見 X 完全相同、Y 卻不同，任何確定性模型都不能同時答對。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
examples = [("紅色物體是", "圓"), ("藍色物體是", "方")]
for length in (1, 3, 5):
    print(length, [(prefix[-length:], answer) for prefix, answer in examples])

**如何讀結果：**足夠上下文是必要條件，不是學會規則的保證；增加窗口也增加輸入層參數。

**只改一件事：**只把區別詞放得更遠，找最小可辨識窗口。
